* Desc: Plot cost of living index across the world
* Data: https://www.numbeo.com/cost-of-living/rankings.jsp


In [ ]:
import os
os.environ['PROJ_LIB'] = '${HOME}/miniconda3/share/proj'

import polars as pl
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
from config import Continent, european_city_coords

In [ ]:
INPUT_FILE = './data/numbeo_cost_of_living_by_cities_2026.csv'
assert INPUT_FILE is not None, f'Input file {INPUT_FILE} not found!' \
                                'Get it from https://www.numbeo.com/cost-of-living/rankings.jsp'

# Load data with Polars
df = pl.read_csv(INPUT_FILE)
print(f'Loaded {len(df)} cities')
df.head()

## Data Cleaning

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.null_count())
print('\nData types:')
print(df.dtypes)

# Check data quality
print('\nFirst few rows:')
df.head(10)

In [ ]:
# Clean city and country columns
# Some cities have state information (e.g., 'Jersey City, NJ, United States')
# Extract just city name and country

df = df.with_columns([
    # Clean city: take first part before comma if comma exists
    pl.col('City').str.strip_chars().str.split(',').list.first().alias('City_Clean'),
    # Clean country: just strip whitespace
    pl.col('Country').str.strip_chars().alias('Country_Clean')
])

print('Cleaned city/country data:')
df.select(['City', 'Country', 'City_Clean', 'Country_Clean']).head(10)

In [ ]:
# Drop rows with missing critical data
df_clean = df.drop_nulls(subset=['Cost_of_Living', 'Cost_of_Living_plus_Rent'])

print(f'Rows after removing missing values: {len(df_clean)} (removed {len(df) - len(df_clean)})')

# Check numeric columns - Polars handles types automatically
numeric_cols = ['Cost_of_Living', 'Rent', 'Cost_of_Living_plus_Rent', 
                'Groceries', 'Restaurant_Price', 'Local_Purchasing_Power']

print('\nStatistics of cleaned data:')
df_clean.select(numeric_cols).describe()

## Filter European Cities

In [ ]:
# Define European countries
european_countries = [
    'Albania', 'Andorra', 'Austria', 'Belarus', 'Belgium', 'Bosnia And Herzegovina',
    'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic', 'Denmark', 'Estonia',
    'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland',
    'Italy', 'Kosovo (Disputed Territory)', 'Latvia', 'Lithuania', 'Luxembourg',
    'Malta', 'Moldova', 'Monaco', 'Montenegro', 'Netherlands', 'North Macedonia',
    'Norway', 'Poland', 'Portugal', 'Romania', 'Russia', 'Serbia', 'Slovakia',
    'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Turkey', 'Ukraine', 'United Kingdom'
]

# Filter for European cities
df_europe = df_clean.filter(pl.col('Country_Clean').is_in(european_countries))

print(f'Found {len(df_europe)} European cities out of {len(df_clean)} total cities')
print('\nEuropean cities:')
df_europe.select(['City_Clean', 'Country_Clean', 'Cost_of_Living_plus_Rent']).sort('Cost_of_Living_plus_Rent', descending=True)

In [ ]:
# Use pre-built city coordinates dataset from config
# Create full location string for better geocoding accuracy
df_europe_clean = df_europe.with_columns([
    (pl.col('City_Clean') + ', ' + pl.col('Country_Clean')).alias('Location')
])

# Convert to Polars DataFrame
city_coords_df = pl.DataFrame({
    'Location': list(european_city_coords.keys()),
    'Latitude': [coord[0] for coord in european_city_coords.values()],
    'Longitude': [coord[1] for coord in european_city_coords.values()]
})

# Merge with df_europe_clean
df_europe_clean_coords = df_europe_clean.join(
    city_coords_df,
    on='Location',
    how='left'
)

# Check how many cities were matched
matched = df_europe_clean_coords.filter(pl.col('Latitude').is_not_null())
unmatched = df_europe_clean_coords.filter(pl.col('Latitude').is_null())

print(f'Matched {len(matched)} out of {len(df_europe_clean)} European cities')
if len(unmatched) > 0:
    print(f'\nUnmatched cities ({len(unmatched)}):')
    print(unmatched['Location'].to_list()[:20])  # Show first 20 unmatched

# Remove rows without coordinates
df_europe_clean_coords = df_europe_clean_coords.drop_nulls(subset=['Latitude', 'Longitude'])
print(f'\nFinal dataset has {len(df_europe_clean_coords)} cities with coordinates')
df_europe_clean_coords.select(['City_Clean', 'Country_Clean', 'Latitude', 'Longitude']).head()

In [ ]:
# Create alias for compatibility with rest of notebook
df_geo = df_europe_clean_coords
print(f'Final dataset has {len(df_geo)} cities with geocodes')

## Map with Proportional Circles for Europe

#### Normalize by Cost of Living + Rent Index

In [ ]:
# Set top city 100, and normalize by Cost of Living + Rent Index
TOP_CITY_COST = 100.0
df_geo_norm = df_geo.with_columns([
    (pl.col('Cost_of_Living_plus_Rent') / pl.col('Cost_of_Living_plus_Rent').max() * TOP_CITY_COST).alias('Normalized_Cost_of_Living_plus_Rent')
])

### Limit to famous cities only

In [ ]:
cities_selected = [
    'Aalborg', 'Aarhus', 'Amsterdam', 'Ankara', 'Athens',
    'Barcelona', 'Bari', 'Belgrade', 'Bergen', 'Berlin', 'Bratislava', 'Brussels', 'Bucharest', 'Budapest',
    'Cardiff', 'Chisinau', 'Copenhagen',
    'Dublin',
    'Edinburgh',
    'Florence', 'Frankfurt',
    'Geneva', 'Gothenburg',
    'Hamburg', 'Helsinki',
    'Istanbul',
    'Kyiv', 'Krakow',
    'Lisbon', 'London', 'Ljubljana', 'Luxembourg', 'Lyon',
    'Madrid', 'Manchester', 'Marseille', 'Milan', 'Minsk', 'Moscow', 'Munich',
    'Naples', 'Nice',
    'Odense', 'Oslo',
    'Paris', 'Porto', 'Prague',
    'Reykjavik', 'Riga', 'Rome',
    'Sarajevo', 'Skopje', 'Seville (Sevilla)', 'Sofia', 'Saint Petersburg', 'Stavanger', 'Stockholm', 'Strasbourg',
    'Tallinn', 'Thessaloniki', 'Tirana', 'Toulouse', 'Trondheim', 'Turin', 'Turku',
    'Valencia', 'Vienna', 'Vilnius',
    'Warsaw',
    'Uppsala',
    'Zagreb', 'Zurich'
]
df_geo_norm_selected = df_geo_norm.filter(pl.col('City_Clean').is_in(cities_selected))

In [ ]:
# Create a Basemap visualization for Europe with proportional circles
SHAPE_FILE = './data/natural_earth/ne_10m_admin_0_countries'
RESOLUTION = 'i'  # i = intermediate resolution
N_FIRST_CITIES = 500  # Number of top cities to plot for clarity
FIGSIZE = (20, 12)

# Get Europe configuration
region = Continent.europe
region_llcrnrlon = region['llcrnrlon']
region_llcrnrlat = region['llcrnrlat']
region_urcrnrlon = region['urcrnrlon'] - 15.0 # Adjust to eastern boundary
region_urcrnrlat = region['urcrnrlat']
region_lat0 = region['lat_0']
region_lon0 = region['lon_0']
region_lat1 = region['lat_1']
region_lat2 = region['lat_2']

# Create figure
fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_subplot(111)

# Create Basemap for Europe
m = Basemap(
    llcrnrlon=region_llcrnrlon,
    llcrnrlat=region_llcrnrlat,
    urcrnrlon=region_urcrnrlon,
    urcrnrlat=region_urcrnrlat,
    lat_0=region_lat0,
    lon_0=region_lon0,
    lat_1=region_lat1,
    lat_2=region_lat2,
    projection='lcc',
    resolution=RESOLUTION,
    area_thresh=1000,
)

# Draw map boundary and features
m.drawmapboundary(fill_color='lightblue', linewidth=0.5)
m.fillcontinents(color='wheat', lake_color='lightblue')
m.drawcountries(linewidth=1.5, color='darkgray')
m.drawcoastlines(linewidth=0.5)

# Read shapefile for political boundaries
m.readshapefile(SHAPE_FILE, 'countries', linewidth=1, color='gray')

# Get data for plotting
lats = df_geo_norm_selected['Latitude'].to_list()
lons = df_geo_norm_selected['Longitude'].to_list()
costs = df_geo_norm_selected['Normalized_Cost_of_Living_plus_Rent'].to_list()
cities = df_geo_norm_selected['City_Clean'].to_list()

# Drop None values from lats, lons, costs, cities
valid_data = [(lat, lon, cost, city) for lat, lon, cost, city in zip(lats, lons, costs, cities)
              if lat is not None and lon is not None and cost is not None and city is not None]
lats, lons, costs, cities = zip(*valid_data)

# Sort by costs descending
sorted_data = sorted(zip(lats, lons, costs, cities), key=lambda x: x[2], reverse=True)
lats, lons, costs, cities = zip(*sorted_data)

# Limit to first N cities for clarity
lats = lats[:N_FIRST_CITIES]
lons = lons[:N_FIRST_CITIES]
costs = costs[:N_FIRST_CITIES]
cities = cities[:N_FIRST_CITIES]

# Convert coordinates to map projection
x, y = m(lons, lats)

# Scale circle sizes
# Use area proportional to cost
sizes = [(c ** 1.3) * 3 for c in costs]

# Create scatter plot with circles
scatter = m.scatter(x, y, s=sizes, alpha=0.7, c=costs,
                    cmap='YlOrRd', edgecolors='black', linewidth=1.5,
                    zorder=5)

# Add value labels inside circles
for xi, yi, cost, city in zip(x, y, costs, cities):
    # Cost value inside circle
    plt.text(xi, yi, f'{cost:.0f}',
             ha='center', va='center', fontsize=7, fontweight='bold',
             zorder=6)
    
    # City name below circle
    if city in ['Uppsala', 'Vienna']:
        plt.text(xi, yi + 60000, city,
             ha='center', va='top', fontsize=6, zorder=6, style='italic',
             color='darkblue',
             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.3'))

    else:
        plt.text(xi, yi - 30000, city,
                ha='center', va='top', fontsize=6, zorder=6, style='italic',
                color='darkblue',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.3'))

# # Add colorbar
# cbar = plt.colorbar(scatter, ax=ax, shrink=0.7, pad=0.02)
# cbar.set_label('Cost of Living + Rent Index', fontsize=12)

# Title
plt.title('Cost of Living + Rent in European Cities (2026)\n(Top city Zurich=100 as the baseline)',
          fontsize=18, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('data/output/europe_cost_of_living_basemap.png', dpi=300, bbox_inches='tight')
print('Saved map as data/output/europe_cost_of_living_basemap.png')
plt.show()

## Summary Statistics

In [ ]:
# Show top 10 most expensive cities
print('Top 10 Cities by Cost of Living:')
print(df_geo.sort('Cost_of_Living', descending=True).select(['City_Clean', 'Country_Clean', 'Cost_of_Living']).head(10))

print('\nTop 10 Cities by Cost of Living + Rent:')
print(df_geo.sort('Cost_of_Living_plus_Rent', descending=True).select(['City_Clean', 'Country_Clean', 'Cost_of_Living_plus_Rent']).head(10))